# Advisor Copilot — RAG-Powered Portfolio Chatbot (v2)

**Module:** AI in Accounting and Finance — Group Final Project (Option A)

**What's new in v2:**
- **Live LLM integration** using Google Gemini (free API — no credit card needed)
- **Interactive chatbot UI** powered by Gradio — runs directly inside Colab
- **Natural language queries** — ask anything about any client's portfolio
- **Deployable** — same code runs on Render as a live public chatbot

**Architecture:**
```
User Question ──► Intent Router ──► Portfolio Analytics (Pandas)
                                 ──► RAG Retriever (FAISS + Sentence-Transformers)
                                         │
                                         ▼
                              Google Gemini LLM + Prompt Templates
                                         │
                                         ▼
                              Grounded, Cited Response in Chat UI
```


## 1. Install Dependencies & Setup

Run this cell first. It installs everything needed and sets up the environment.


In [ ]:
# ============================================================
# STEP 1: Install all dependencies
# ============================================================
!pip install -q sentence-transformers faiss-cpu pandas numpy google-generativeai gradio tabulate

import os, json, textwrap, math, re
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 120)
print("✅ All packages installed and imported.")


## 2. Set Your Google Gemini API Key (FREE)

**How to get your free key (takes 30 seconds):**
1. Go to https://aistudio.google.com/apikey
2. Click "Create API Key"
3. Copy it and paste below

The free tier gives you 15 requests/minute — more than enough for this demo.


In [ ]:
# ============================================================
# STEP 2: Enter your Gemini API key
# ============================================================
# OPTION A: Paste directly (easy for demo)
GEMINI_API_KEY = ""  # <-- paste your key between the quotes

# OPTION B: Use Colab Secrets (more secure)
# Go to the 🔑 icon in the left sidebar → add secret named GOOGLE_API_KEY
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
        print("✅ Loaded API key from Colab Secrets.")
    except Exception:
        pass

if not GEMINI_API_KEY:
    print("⚠️  No API key found. The chatbot will run in DEMO MODE with template responses.")
    print("   To get live AI responses, get a free key at: https://aistudio.google.com/apikey")
    USE_LLM = False
else:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    USE_LLM = True
    print("✅ Gemini API key configured successfully!")


## 3. Load Client Data

Upload the `data/` folder to Colab, or if you mounted Google Drive, adjust the path below.


In [ ]:
# ============================================================
# STEP 3: Load the client portfolio data
# ============================================================
# If running in Colab: upload the data folder or mount Drive
# If running locally: just make sure the data/ folder is next to this notebook

import os

DATA_DIR = "data"

# Auto-detect: check if data folder exists, if not try to find it
if not os.path.exists(DATA_DIR):
    # Try Google Drive mount
    if os.path.exists("/content/drive/MyDrive"):
        for root, dirs, files in os.walk("/content/drive/MyDrive"):
            if "clients.csv" in files and "holdings.csv" in files:
                DATA_DIR = root
                print(f"Found data at: {DATA_DIR}")
                break
    # If still not found, create sample data inline
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR, exist_ok=True)
        print("Creating data folder with embedded sample data...")

        # clients.csv
        with open(f"{DATA_DIR}/clients.csv", "w") as f:
            f.write("""client_id,name,age,risk_profile,model_portfolio,goal,inception_date,base_currency
C001,Emily Tan,42,Balanced,Balanced_60_40,Retirement at 60,2019-03-15,USD
C002,Rajesh Kumar,58,Conservative,Conservative_30_70,Capital preservation,2015-07-01,USD
C003,Sofia Martinez,31,Aggressive,Growth_90_10,Long-term wealth accumulation,2021-11-20,USD
C004,Daniel Lee,49,Balanced,Balanced_60_40,Children education fund,2017-05-10,USD
C005,Aisha Rahman,65,Conservative,Conservative_30_70,Income generation,2012-02-28,USD""")

        # holdings.csv
        with open(f"{DATA_DIR}/holdings.csv", "w") as f:
            f.write("""client_id,ticker,asset_class,sector,quantity,avg_cost,current_price,currency
C001,VTI,Equity,US Broad Market,120,195.40,248.70,USD
C001,VXUS,Equity,Intl Developed,80,54.10,62.30,USD
C001,BND,Fixed Income,US Aggregate,250,78.20,72.90,USD
C001,VWO,Equity,Emerging Markets,60,41.30,45.80,USD
C002,BND,Fixed Income,US Aggregate,600,80.50,72.90,USD
C002,TLT,Fixed Income,Long Treasury,150,110.20,92.40,USD
C002,VTI,Equity,US Broad Market,40,190.00,248.70,USD
C002,VNQ,Real Estate,US REITs,35,88.50,84.20,USD
C003,QQQ,Equity,US Tech,50,310.00,478.30,USD
C003,VTI,Equity,US Broad Market,80,210.00,248.70,USD
C003,ARKK,Equity,Innovation,100,45.00,52.40,USD
C003,VWO,Equity,Emerging Markets,120,42.00,45.80,USD
C004,VTI,Equity,US Broad Market,100,200.00,248.70,USD
C004,VXUS,Equity,Intl Developed,90,55.00,62.30,USD
C004,BND,Fixed Income,US Aggregate,300,79.00,72.90,USD
C004,VNQ,Real Estate,US REITs,25,90.00,84.20,USD
C005,BND,Fixed Income,US Aggregate,700,82.00,72.90,USD
C005,SCHD,Equity,US Dividend,90,72.00,82.10,USD
C005,VNQ,Real Estate,US REITs,40,89.00,84.20,USD
C005,TLT,Fixed Income,Long Treasury,120,108.00,92.40,USD""")

        # transactions.csv
        with open(f"{DATA_DIR}/transactions.csv", "w") as f:
            f.write("""client_id,date,action,ticker,quantity,price,notes
C001,2026-01-15,BUY,VTI,20,240.10,Quarterly contribution
C001,2026-02-10,SELL,VWO,10,44.20,Rebalance EM
C001,2026-03-05,BUY,BND,30,73.50,Rebalance to target
C002,2026-01-20,BUY,BND,50,73.80,Increase bond weight
C002,2026-02-28,SELL,VTI,5,245.00,Trim equity
C003,2026-01-05,BUY,QQQ,10,470.00,Add tech exposure
C003,2026-02-15,BUY,ARKK,20,50.10,Speculative add-on
C003,2026-03-20,SELL,VWO,15,45.00,Profit taking
C004,2026-01-25,BUY,VTI,15,242.00,Education fund top-up
C004,2026-03-01,BUY,BND,25,73.00,Rebalance
C005,2026-01-10,BUY,SCHD,10,81.00,Dividend income
C005,2026-02-20,SELL,TLT,10,93.00,Rate outlook shift""")

        # Knowledge base files
        with open(f"{DATA_DIR}/kb_suitability_policy.txt", "w") as f:
            f.write("""DOCUMENT: Client Suitability Policy (Internal, v3.2, Jan 2026)

1. Risk profile classifications.
The firm classifies clients into three risk profiles: Conservative, Balanced, and Aggressive. Each profile is mapped to a target equity/fixed-income split with an allowable drift band of +/- 5 percentage points before rebalancing is required.

2. Target allocations by profile.
Conservative clients target 30% equities and 70% fixed income / cash. Balanced clients target 60% equities and 40% fixed income / cash. Aggressive clients target 90% equities and 10% fixed income / cash. Real estate (REITs) counts as equity exposure for suitability purposes.

3. Concentration limits.
No single position may exceed 10% of total portfolio value for Conservative clients, 15% for Balanced, and 20% for Aggressive. Sector concentration in any one GICS sector must remain below 25% for all profiles. Thematic/innovation ETFs (e.g., ARKK) are capped at 5% for Conservative, 10% for Balanced, and 15% for Aggressive.

4. Product eligibility.
Conservative clients may not hold leveraged ETFs, single-stock options, or crypto products. Balanced clients may hold up to 5% in alternative assets. Aggressive clients may hold up to 15% in alternatives including crypto ETPs.

5. Rebalancing rules.
Portfolios are reviewed quarterly. If any asset class drifts more than 5 percentage points from the model, a rebalance proposal must be drafted. Tax-loss harvesting opportunities should be flagged when unrealized losses exceed USD 2,000 on a single position.

6. Disclosure and approval.
Every client communication generated by the Advisor Copilot must be reviewed and approved by a licensed Relationship Manager (RM) before being sent. No automated output may be released to clients without human sign-off.""")

        with open(f"{DATA_DIR}/kb_model_portfolios.txt", "w") as f:
            f.write("""DOCUMENT: Model Portfolios and House Views (Investment Committee, Q1 2026)

Conservative_30_70 model portfolio: 20% US Aggregate Bonds (BND), 25% Long Treasuries (TLT), 15% US Broad Equity (VTI), 10% US Dividend (SCHD), 10% US REITs (VNQ), 20% Cash and Money Market. Target yield: 3.8%. Target volatility: 6%.

Balanced_60_40 model portfolio: 35% US Broad Equity (VTI), 15% International Developed (VXUS), 5% Emerging Markets (VWO), 5% US REITs (VNQ), 25% US Aggregate Bonds (BND), 10% Short Treasuries (SHY), 5% Cash. Target return: 6-7%. Target volatility: 10%.

Growth_90_10 model portfolio: 45% US Broad Equity (VTI), 20% US Tech (QQQ), 10% International Developed (VXUS), 10% Emerging Markets (VWO), 5% Innovation (ARKK), 10% Short Treasuries (SHY). Target return: 8-10%. Target volatility: 15%.

House view, Q1 2026: The Investment Committee maintains a neutral stance on US equities and a modest overweight on International Developed markets due to relative valuation. Duration is kept neutral to slightly short given sticky services inflation. Emerging markets are downgraded to underweight due to currency pressure. Innovation ETFs such as ARKK should only be recommended where thematic exposure fits the client goal and within policy caps.

Rebalancing cadence: Quarterly with an interim mid-quarter check when volatility (VIX) exceeds 25.""")

        with open(f"{DATA_DIR}/kb_research_notes.txt", "w") as f:
            f.write("""DOCUMENT: Internal Research Notes (Research Desk, March 2026)

Macro outlook. Global growth is forecast at 2.9% for 2026, with US at 2.1% and Eurozone at 1.2%. Inflation in the US is expected to glide toward 2.6% by year end. The Federal Reserve is projected to cut rates by 50 bps over the next twelve months, supporting intermediate duration bonds.

Equity view. Earnings growth for the S&P 500 is expected at 9% year over year. Mega-cap technology remains resilient but valuation is stretched; the research desk prefers broad market exposure (VTI) over concentrated tech (QQQ) for new contributions. International developed markets offer more attractive valuations with a forward P/E below long-term averages.

Fixed income view. The team recommends neutral duration with a tilt toward intermediate Treasuries. Long-duration TLT is expected to rally if growth softens but should not exceed 20% of the fixed income sleeve. Credit spreads are tight; investment-grade preferred over high yield.

Thematic view. Innovation ETFs (ARKK) face idiosyncratic drawdown risk. Position sizing should be strictly capped per the suitability policy. Real estate (VNQ) is attractive for income-oriented clients given the yield pickup and discount to NAV.

Client communication guidance. When explaining recent performance, reference the house view and attribute sources clearly. Avoid forward-looking guarantees. Frame rebalancing as bringing the portfolio back to the client's pre-agreed strategic allocation rather than market timing.""")

        with open(f"{DATA_DIR}/kb_product_factsheets.txt", "w") as f:
            f.write("""DOCUMENT: Product Factsheets (Curated, Jan 2026)

VTI - Vanguard Total Stock Market ETF. Tracks the CRSP US Total Market Index. Expense ratio 0.03%. Holds more than 3,700 US stocks across large, mid, and small cap. Role: core US equity exposure.

VXUS - Vanguard Total International Stock ETF. Tracks FTSE Global All Cap ex US Index. Expense ratio 0.08%. Role: diversified ex-US equity exposure.

VWO - Vanguard FTSE Emerging Markets ETF. Tracks FTSE Emerging Markets Index. Expense ratio 0.08%. Higher volatility. Role: emerging markets allocation.

BND - Vanguard Total Bond Market ETF. Tracks Bloomberg US Aggregate Float Adjusted Index. Expense ratio 0.03%. Duration approximately 6 years. Role: core US fixed income.

TLT - iShares 20+ Year Treasury Bond ETF. Long-duration Treasuries, duration approximately 17 years. High rate sensitivity. Role: tactical duration, hedge against growth shocks.

SHY - iShares 1-3 Year Treasury Bond ETF. Short duration, low volatility. Role: cash-like stability.

SCHD - Schwab US Dividend Equity ETF. Quality dividend screen. Yield approximately 3.4%. Role: equity income.

VNQ - Vanguard Real Estate ETF. US REIT exposure. Yield approximately 4%. Role: real asset and income diversification.

QQQ - Invesco Nasdaq 100 ETF. 100 largest non-financial Nasdaq stocks, tech-heavy. Role: growth tilt, counts against sector concentration limits.

ARKK - ARK Innovation ETF. Actively managed disruptive innovation theme. High volatility and drawdown risk. Role: satellite thematic position subject to strict caps.""")

        print("✅ All sample data files created.")

# Load the data
clients = pd.read_csv(f"{DATA_DIR}/clients.csv")
holdings = pd.read_csv(f"{DATA_DIR}/holdings.csv")
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv")

print(f"\n✅ Data loaded: {len(clients)} clients, {len(holdings)} holdings, {len(transactions)} transactions")
print("\nClients:")
display(clients[['client_id', 'name', 'risk_profile', 'model_portfolio', 'goal']])


## 4. Portfolio Analytics Engine

Deterministic calculations — market value, P&L, allocation drift, concentration checks. These numbers ground the LLM so it never has to guess.


In [ ]:
# ============================================================
# STEP 4: Portfolio analytics functions
# ============================================================

def portfolio_snapshot(client_id: str) -> dict:
    """Compute a structured snapshot for one client."""
    c = clients[clients.client_id == client_id].iloc[0].to_dict()
    h = holdings[holdings.client_id == client_id].copy()
    h["market_value"] = h.quantity * h.current_price
    h["cost_basis"]   = h.quantity * h.avg_cost
    h["pnl"]          = h.market_value - h.cost_basis
    h["pnl_pct"]      = (h.pnl / h.cost_basis) * 100

    total_mv = h.market_value.sum()
    h["weight_pct"] = (h.market_value / total_mv) * 100

    by_class = (h.groupby("asset_class")["market_value"]
                  .sum()
                  .apply(lambda v: round(v / total_mv * 100, 1))
                  .to_dict())

    targets = {
        "Conservative_30_70": {"Equity": 15, "Fixed Income": 65, "Real Estate": 10, "Cash": 10},
        "Balanced_60_40":     {"Equity": 55, "Fixed Income": 35, "Real Estate": 5,  "Cash": 5},
        "Growth_90_10":       {"Equity": 85, "Fixed Income": 10, "Real Estate": 0,  "Cash": 5},
    }
    target = targets.get(c["model_portfolio"], {})
    drift = {k: round(by_class.get(k, 0) - target.get(k, 0), 1) for k in target}

    recent_tx = (transactions[transactions.client_id == client_id]
                 .sort_values("date")
                 .tail(5)
                 .to_dict(orient="records"))

    # Concentration check
    concentration_flags = []
    limits = {"Conservative": 10, "Balanced": 15, "Aggressive": 20}
    limit = limits.get(c["risk_profile"], 15)
    for _, row in h.iterrows():
        if row["weight_pct"] > limit:
            concentration_flags.append(
                f"{row['ticker']} is {row['weight_pct']:.1f}% of portfolio (limit: {limit}%)"
            )

    return {
        "client": c,
        "total_market_value": round(total_mv, 2),
        "total_pnl": round(h.pnl.sum(), 2),
        "total_pnl_pct": round(h.pnl.sum() / h.cost_basis.sum() * 100, 2),
        "allocation_pct_by_class": by_class,
        "target_allocation_pct": target,
        "drift_vs_target_pp": drift,
        "concentration_flags": concentration_flags,
        "positions": h[["ticker","asset_class","sector","quantity","avg_cost",
                        "current_price","market_value","weight_pct","pnl","pnl_pct"]]
                     .round(2).to_dict(orient="records"),
        "recent_transactions": recent_tx,
    }

def get_client_id_from_name(name_query: str) -> str:
    """Fuzzy match a client name to their ID."""
    name_query = name_query.lower().strip()
    for _, row in clients.iterrows():
        if name_query in row['name'].lower() or name_query == row['client_id'].lower():
            return row['client_id']
    return None

# Test it
snap = portfolio_snapshot("C001")
print(f"✅ Portfolio engine ready. Test: Emily Tan's portfolio = ${snap['total_market_value']:,.2f}")
print(f"   P&L: {snap['total_pnl_pct']:+.2f}%  |  Drift flags: {snap['drift_vs_target_pp']}")


## 5. Build the RAG Knowledge Base (Vector Store)

Load internal documents → chunk → embed with sentence-transformers → index in FAISS.


In [ ]:
# ============================================================
# STEP 5: Build the RAG retriever
# ============================================================
from sentence_transformers import SentenceTransformer
import faiss

KB_FILES = [
    ("Suitability Policy",   f"{DATA_DIR}/kb_suitability_policy.txt"),
    ("Model Portfolios",     f"{DATA_DIR}/kb_model_portfolios.txt"),
    ("Research Notes",       f"{DATA_DIR}/kb_research_notes.txt"),
    ("Product Factsheets",   f"{DATA_DIR}/kb_product_factsheets.txt"),
]

def chunk_text(text, size=400, overlap=80):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+size]))
        i += size - overlap
    return chunks

docs = []
for source, path in KB_FILES:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    for j, ch in enumerate(chunk_text(raw)):
        docs.append({"source": source, "chunk_id": j, "text": ch})

print(f"Loaded {len(docs)} chunks from {len(KB_FILES)} documents.")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode([d["text"] for d in docs], normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.asarray(embeddings, dtype="float32"))
print(f"✅ FAISS index built: {index.ntotal} vectors, dim = {embeddings.shape[1]}")

def retrieve(query: str, k: int = 4):
    """Return top-k knowledge-base chunks for a query."""
    q_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_vec, k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        d = docs[idx]
        hits.append({"source": d["source"], "chunk_id": d["chunk_id"],
                     "score": float(score), "text": d["text"]})
    return hits

# Test retrieval
print("\nTest retrieval: 'concentration limit for Balanced client'")
for h in retrieve("concentration limit for Balanced client", k=2):
    print(f"  [{h['source']}] score={h['score']:.3f}")


## 6. LLM Integration — Google Gemini

This is the core AI engine. It calls Google Gemini to generate grounded responses using the retrieved context and portfolio data.


In [ ]:
# ============================================================
# STEP 6: LLM wrapper with Gemini + fallback
# ============================================================

SYSTEM_PROMPT = """You are Advisor Copilot, an AI assistant for licensed Relationship Managers (RMs) at a wealth management firm.

RULES YOU MUST FOLLOW:
1. Only use the CLIENT DATA and RETRIEVED CONTEXT provided. Never invent numbers.
2. If a fact is not in the context, say "not available in the provided sources."
3. Cite sources inline like [Source: Document Name] after claims from the knowledge base.
4. Never make forward-looking guarantees about returns.
5. Be specific with numbers — use dollar amounts, percentages, and position names.
6. Structure responses clearly with sections when appropriate.
7. For trading strategies, always reference the firm's house view and suitability policy.
8. End client-facing drafts with: "This communication has been reviewed by your advisor. Past performance is not indicative of future results."

You help RMs with:
- Portfolio summaries and performance reviews
- Suitability and compliance checks
- Rebalancing recommendations and trading strategies
- Client-ready communications
- Any questions about client accounts, positions, or firm policies
"""

def call_llm(system: str, user: str) -> str:
    """Call Google Gemini, or fall back to template mode."""
    if USE_LLM:
        try:
            model = genai.GenerativeModel("gemini-2.0-flash")
            response = model.generate_content(
                f"SYSTEM INSTRUCTIONS:\n{system}\n\nUSER REQUEST:\n{user}",
                generation_config=genai.types.GenerationConfig(
                    temperature=0.2,
                    max_output_tokens=2048,
                )
            )
            return response.text.strip()
        except Exception as e:
            return f"[Gemini API error: {e}]\n\n" + _template_fallback(user)
    return _template_fallback(user)

def _template_fallback(user: str) -> str:
    """Structured template response when no API key is available."""
    # Extract some useful info from the prompt
    lines = user.split("\n")
    task_line = ""
    for line in lines:
        if line.strip().startswith("TASK:") or line.strip().startswith("QUESTION:"):
            task_line = line.strip()
            break

    return (
        "**DEMO MODE** (no API key configured)\n\n"
        "In production, the Gemini LLM would generate a detailed, grounded response here "
        "using the retrieved context and client data.\n\n"
        f"**Request:** {task_line}\n\n"
        "**What the LLM would do:**\n"
        "- Analyse the client data provided (portfolio values, allocations, drift)\n"
        "- Cross-reference with retrieved knowledge base documents\n"
        "- Generate a cited, specific response following the system prompt rules\n\n"
        "To enable live AI responses, add your free Gemini API key in Step 2 above."
    )

print("✅ LLM engine ready." + (" (Gemini connected)" if USE_LLM else " (Demo mode — add API key for live responses)"))


## 7. Chatbot Brain — Intent Detection & Response Generation

The chatbot understands natural language questions, identifies which client you're asking about, and routes to the right analysis pipeline.


In [ ]:
# ============================================================
# STEP 7: The chatbot brain
# ============================================================

def format_context(hits):
    return "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)

def detect_client(message: str) -> str:
    """Try to find a client ID or name in the user's message."""
    msg = message.lower()

    # Check for direct client IDs
    for cid in clients.client_id:
        if cid.lower() in msg:
            return cid

    # Check for names
    for _, row in clients.iterrows():
        name_parts = row['name'].lower().split()
        for part in name_parts:
            if len(part) > 2 and part in msg:
                return row['client_id']

    return None

def detect_intent(message: str) -> str:
    """Classify what the user is asking for."""
    msg = message.lower()

    if any(w in msg for w in ['summary', 'overview', 'tell me about', 'how is', 'performance', 'snapshot']):
        return 'summary'
    if any(w in msg for w in ['suitable', 'suitability', 'compliance', 'breach', 'compliant', 'aligned', 'alignment']):
        return 'suitability'
    if any(w in msg for w in ['rebalance', 'rebalancing', 'trade', 'trading', 'buy', 'sell', 'strategy', 'recommendation']):
        return 'rebalance'
    if any(w in msg for w in ['email', 'draft', 'letter', 'client-ready', 'communication', 'write to']):
        return 'client_email'
    if any(w in msg for w in ['all client', 'book', 'batch', 'everyone', 'all portfolios', 'full book']):
        return 'batch_review'
    if any(w in msg for w in ['policy', 'rule', 'limit', 'concentration', 'guideline']):
        return 'policy_question'
    if any(w in msg for w in ['market', 'outlook', 'house view', 'macro', 'research']):
        return 'market_view'

    return 'general'

def build_client_list_response():
    """Show all clients with key metrics."""
    rows = []
    for cid in clients.client_id:
        s = portfolio_snapshot(cid)
        max_drift = max((abs(v) for v in s["drift_vs_target_pp"].values()), default=0)
        rows.append({
            "Client": f"{s['client']['name']} ({cid})",
            "Risk Profile": s['client']['risk_profile'],
            "Portfolio Value": f"${s['total_market_value']:,.0f}",
            "P&L": f"{s['total_pnl_pct']:+.1f}%",
            "Max Drift": f"{max_drift:.1f}pp",
            "Needs Review": "⚠️ YES" if max_drift > 5 else "✅ OK"
        })

    df = pd.DataFrame(rows)
    table = df.to_string(index=False)
    return f"**Full Book Overview**\n\n```\n{table}\n```\n\nAsk me about any specific client for a deeper dive!"

def process_message(message: str, history: list) -> str:
    """Main chatbot handler — processes user message and returns response."""

    # Special commands
    if message.strip().lower() in ['help', '/help', 'what can you do']:
        return (
            "**I'm your Advisor Copilot!** Here's what I can help with:\n\n"
            "**Client Analysis** — Ask about any client by name or ID:\n"
            "- _'Give me a summary of Emily Tan's portfolio'_\n"
            "- _'Is C002 suitable? Any compliance issues?'_\n"
            "- _'What trades should we make for Sofia Martinez?'_\n"
            "- _'Draft an email for Daniel Lee about his review'_\n\n"
            "**Book-Level** — See all clients at once:\n"
            "- _'Show me all clients'_ or _'Who needs a review?'_\n\n"
            "**Policy & Market** — Ask about firm guidelines:\n"
            "- _'What are the concentration limits?'_\n"
            "- _'What is the current house view on equities?'_\n\n"
            "**Our clients:** Emily Tan (C001), Rajesh Kumar (C002), "
            "Sofia Martinez (C003), Daniel Lee (C004), Aisha Rahman (C005)"
        )

    if message.strip().lower() in ['clients', 'list clients', 'show all clients', 'all clients', 'book']:
        return build_client_list_response()

    # Detect intent and client
    intent = detect_intent(message)
    client_id = detect_client(message)

    # If asking about a specific client
    if client_id:
        snap = portfolio_snapshot(client_id)
        client_name = snap['client']['name']

        if intent == 'summary':
            query = f"performance review for {snap['client']['risk_profile']} client house view"
            hits = retrieve(query, k=3)
            user_prompt = f"""TASK: Write a comprehensive portfolio summary for this client.
Include: total value, P&L, asset allocation, recent activity, and any notable observations.

CLIENT DATA:
{json.dumps({k: snap[k] for k in ['client','total_market_value','total_pnl','total_pnl_pct',
             'allocation_pct_by_class','drift_vs_target_pp','concentration_flags','recent_transactions']}, indent=2, default=str)}

POSITIONS:
{json.dumps(snap['positions'], indent=2, default=str)}

RETRIEVED CONTEXT:
{format_context(hits)}"""

        elif intent == 'suitability':
            query = f"suitability policy concentration limits {snap['client']['risk_profile']}"
            hits = retrieve(query, k=4)
            user_prompt = f"""TASK: Perform a full suitability and compliance check for this client.
Check: (1) allocation vs model portfolio drift, (2) concentration limits, (3) product eligibility.
Give a clear PASS/FAIL verdict with specific numbers.

CLIENT DATA:
{json.dumps({k: snap[k] for k in ['client','allocation_pct_by_class','target_allocation_pct',
             'drift_vs_target_pp','concentration_flags','positions']}, indent=2, default=str)}

RETRIEVED CONTEXT:
{format_context(hits)}"""

        elif intent == 'rebalance':
            query = f"rebalancing recommendation {snap['client']['model_portfolio']} house view trading"
            hits = retrieve(query, k=4)
            user_prompt = f"""TASK: Propose a detailed rebalancing plan with specific trading strategies.
Include: (1) BUY/SELL recommendations with dollar amounts, (2) rationale citing house view,
(3) priority order, (4) tax-loss harvesting opportunities if any.

CLIENT DATA:
{json.dumps({k: snap[k] for k in ['client','total_market_value','allocation_pct_by_class',
             'target_allocation_pct','drift_vs_target_pp','concentration_flags','positions']}, indent=2, default=str)}

RETRIEVED CONTEXT:
{format_context(hits)}"""

        elif intent == 'client_email':
            query = "client communication guidance rebalancing portfolio review"
            hits = retrieve(query, k=3)
            user_prompt = f"""TASK: Draft a professional, warm client-ready email (max 200 words).
Explain the portfolio review findings and proposed next steps in plain English.
End with the mandatory compliance disclaimer.

CLIENT DATA:
{json.dumps({k: snap[k] for k in ['client','total_pnl_pct','allocation_pct_by_class',
             'drift_vs_target_pp']}, indent=2, default=str)}

RETRIEVED CONTEXT:
{format_context(hits)}"""

        else:
            # General question about a specific client
            query = message
            hits = retrieve(query, k=4)
            user_prompt = f"""QUESTION: {message}

Answer this question using the client data and retrieved context below.
Be specific and cite sources.

CLIENT DATA:
{json.dumps(snap, indent=2, default=str)}

RETRIEVED CONTEXT:
{format_context(hits)}"""

        response = call_llm(SYSTEM_PROMPT, user_prompt)
        return f"**{client_name} ({client_id})** — {intent.replace('_', ' ').title()}\n\n{response}"

    # Batch review
    elif intent == 'batch_review':
        return build_client_list_response()

    # Policy or market questions (no specific client)
    elif intent in ['policy_question', 'market_view', 'general']:
        hits = retrieve(message, k=4)
        user_prompt = f"""QUESTION: {message}

Answer using ONLY the retrieved context below. Cite sources.

RETRIEVED CONTEXT:
{format_context(hits)}

AVAILABLE CLIENTS FOR REFERENCE:
{clients[['client_id','name','risk_profile','model_portfolio']].to_string(index=False)}"""
        response = call_llm(SYSTEM_PROMPT, user_prompt)
        return response

    # Fallback
    else:
        return (
            "I'm not sure which client you're asking about. Try:\n"
            "- **'Summary for Emily Tan'**\n"
            "- **'Is C003 suitable?'**\n"
            "- **'Trading strategy for Rajesh Kumar'**\n"
            "- **'Show all clients'**\n\n"
            "Type **help** to see everything I can do!"
        )

print("✅ Chatbot brain ready!")


## 8. Quick Test (Non-Interactive)

Let's verify everything works before launching the UI.


In [ ]:
# ============================================================
# STEP 8: Quick test — run these to verify the pipeline works
# ============================================================
print("=" * 70)
print("TEST 1: Portfolio Summary for Emily Tan")
print("=" * 70)
result = process_message("Give me a summary of Emily Tan's portfolio", [])
print(result)

print("\n" + "=" * 70)
print("TEST 2: All clients overview")
print("=" * 70)
result = process_message("Show all clients", [])
print(result)


## 9. Launch the Interactive Chatbot UI

This launches a **Gradio chatbot** right inside Colab. You can chat naturally with the copilot.

When you run this cell, a chat interface will appear below. You can also click the **public URL** to share it temporarily with anyone.


In [ ]:
# ============================================================
# STEP 9: Launch the Gradio chatbot interface
# ============================================================
import gradio as gr

def chat_fn(message, history):
    """Gradio chat handler."""
    response = process_message(message, history)
    return response

EXAMPLE_QUERIES = [
    "Show all clients",
    "Give me a full summary of Emily Tan's portfolio",
    "Is Rajesh Kumar's portfolio compliant with suitability policy?",
    "What trading strategy do you recommend for Sofia Martinez?",
    "Draft a client email for Daniel Lee about his portfolio review",
    "Who needs a rebalance review?",
    "What are the concentration limits for Balanced clients?",
    "What is the current house view on equities?",
    "Analyse C005 Aisha Rahman's portfolio for tax-loss harvesting",
]

demo = gr.ChatInterface(
    fn=chat_fn,
    title="🏦 Advisor Copilot — RAG Portfolio Assistant",
    description=(
        "Ask me about any client's portfolio, suitability, trading strategies, "
        "or firm policies. Type **help** to see all available commands.\n\n"
        "**Clients:** Emily Tan (C001) · Rajesh Kumar (C002) · Sofia Martinez (C003) "
        "· Daniel Lee (C004) · Aisha Rahman (C005)"
    ),
    examples=EXAMPLE_QUERIES,
    theme=gr.themes.Soft(),
    retry_btn="🔄 Retry",
    undo_btn="↩️ Undo",
    clear_btn="🗑️ Clear",
)

# Launch — share=True gives you a public URL for demo/video recording
demo.launch(share=True, debug=False)


## 10. Deploying as a Live Chatbot on Render

To make this a permanent live chatbot (not just in Colab), you can deploy it on **Render** for free.

See the PDF instructions document for the full step-by-step guide, or follow the quick steps below:

### Quick Deploy Steps:
1. Create a GitHub repo and push the `Option_A_Code_v2` folder
2. Sign up at [render.com](https://render.com) (free tier)
3. New → Web Service → Connect your GitHub repo
4. Set:
   - **Build command:** `pip install -r requirements.txt`
   - **Start command:** `python app.py`
   - **Environment variable:** `GOOGLE_API_KEY` = your Gemini key
5. Deploy!

The `app.py` and `requirements.txt` files are included in the `Option_A_Code_v2` folder.


## 11. Evaluation, Risks & Ethics

**Hallucination control.** Every LLM answer is grounded in retrieved context and deterministic portfolio analytics. The system prompt forbids inventing numbers and requires source citations.

**RAG architecture.** Documents are chunked, embedded with sentence-transformers (all-MiniLM-L6-v2), and indexed in FAISS. Top-k retrieval ensures the LLM only sees relevant policy/research context.

**Human-in-the-loop.** No output goes to clients directly. A licensed RM reviews and approves every draft — this is a regulatory requirement, not optional.

**Data governance.** All client data in this demo is synthetic. In production, the copilot would connect to a read-only data mart with row-level access control.

**Technology stack.** Google Gemini (free tier), Sentence-Transformers, FAISS, Gradio, Pandas — all open-source or free-tier, making this reproducible without cost.

**Regulatory compliance.** The copilot flags suitability breaches, enforces concentration limits per policy, and includes mandatory disclaimers on all client-facing content.

---

**Grading rubric mapping:**
- *Coding/technical:* Full RAG pipeline, vector store, LLM integration, interactive chatbot UI, deployable architecture
- *Problem relevance:* Directly solves the RM productivity problem with grounded, policy-compliant AI
- *AI/LLM effectiveness:* Gemini generates contextual, cited responses; template fallback ensures demo always works
- *RAG integration:* FAISS + sentence-transformers retrieval with 4 knowledge base documents
- *Creativity:* Natural language chatbot interface, deployable on Render, batch book review
